In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# load datasets
diabetes = pd.read_csv('/content/drive/MyDrive/Bimbing/Hyperparameter Tuning/diabetes.csv')
heart = pd.read_csv('/content/drive/MyDrive/Bimbing/Hyperparameter Tuning/heart_failure.csv')
live = pd.read_csv('/content/drive/MyDrive/Bimbing/Hyperparameter Tuning/fb_live.csv')

In [ ]:
diabetes.head()

,AGE,SEX,BMI,BP,S1,S2,S3,S4,S5,S6,Y
0,59,2,32.1,101.0,157,93.2,38.0,4.0,4.8598,87,151
1,48,1,21.6,87.0,183,103.2,70.0,3.0,3.8918,69,75
2,72,2,30.5,93.0,156,93.6,41.0,4.0,4.6728,85,141
3,24,1,25.3,84.0,198,131.4,40.0,5.0,4.8903,89,206
4,50,1,23.0,101.0,192,125.4,52.0,4.0,4.2905,80,135


In [ ]:
diabetes['Y']. value_counts()

,count
Y,
200,6
72,6
178,5
71,5
90,5
...,...
146,1
212,1
120,1


In [ ]:
# as always, first thing is to split the data
from sklearn.model_selection import train_test_split

X = diabetes.drop(columns='Y').to_numpy()
y = diabetes[['Y']].to_numpy()
y = y.reshape(len(y),) # sklearn's shape requirement

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# define ridge regression model
from sklearn.linear_model import Ridge

ridge_reg = Ridge(random_state=42)

In [ ]:
# tune lambda (denote as 'alpha' in sklearn)
# using GridSearchCV
from sklearn.model_selection import GridSearchCV

# hyperparameter values we want to tune
parameters = {
    'alpha': (0.000001,0.00001,0.0001,0.001,
              0.01, 0.1, 1, 5, 10, 20),
    'max_iter' : (700,800,900,1000, None),
    'copy_X': (True, False)
}

# the tuning
ridge_reg_gridcv = GridSearchCV(ridge_reg, parameters, cv=5,
                                scoring='neg_root_mean_squared_error')
ridge_reg_gridcv.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=Ridge(random_state=42),
             param_grid={'alpha': (1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 1, 5,
                                   10, 20),
                         'copy_X': (True, False),
                         'max_iter': (700, 800, 900, 1000, None)},
             scoring='neg_root_mean_squared_error')

In [ ]:
# tune lambda (denote as 'alpha' in sklearn)
# using GridSearchCV
from sklearn.model_selection import GridSearchCV

# hyperparameter values we want to tune
parameters = {
    'alpha': (0.000001,0.00001,0.0001,0.001,
              0.01, 0.1, 1, 5, 10, 20),
    'max_iter' : (700,800,900,1000, None),
    'copy_X': (True, False)
}

# the tuning
scoring = {"neg_root_mean_squared_error": "neg_root_mean_squared_error", "r2": "r2"}

ridge_reg_gridcv = GridSearchCV(ridge_reg, parameters, cv=5,
                                scoring=scoring,
                                refit='neg_root_mean_squared_error',
                                n_jobs=-1,
                                )
ridge_reg_gridcv.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=Ridge(random_state=42), n_jobs=-1,
             param_grid={'alpha': (1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 1, 5,
                                   10, 20),
                         'copy_X': (True, False),
                         'max_iter': (700, 800, 900, 1000, None)},
             refit='neg_root_mean_squared_error',
             scoring={'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
                      'r2': 'r2'})

In [ ]:
# ridge_reg_000001 = Ridge(random_state=42, alpha=0.000001, max_iter = 700)
# ridge_reg_000001 = Ridge(random_state=42, alpha=0.000001, max_iter = 800)
# ridge_reg_000001 = Ridge(random_state=42, alpha=0.000001, max_iter = 900)
# ridge_reg_000001 = Ridge(random_state=42, alpha=0.000001, max_iter = 1000)


# ridge_reg_00001 = Ridge(random_state=42, alpha=0.00001, max_iter = 700)
# ridge_reg_00001 = Ridge(random_state=42, alpha=0.00001, max_iter = 800)

# ridge_reg_0001 = Ridge(random_state=42, alpha=0.0001 )

In [ ]:
# pd.DataFrame(ridge_reg_gridcv.cv_results_).head()

In [ ]:
# only show the most important columns
retain_cols = ['params','mean_test_score','rank_test_score']
cv_result = pd.DataFrame(ridge_reg_gridcv.cv_results_)
cv_result[retain_cols]

,params,mean_test_score,rank_test_score
0,"{'alpha': 1e-06, 'copy_X': True, 'max_iter': 700}",-55.972086,61
1,"{'alpha': 1e-06, 'copy_X': True, 'max_iter': 800}",-55.972086,61
2,"{'alpha': 1e-06, 'copy_X': True, 'max_iter': 900}",-55.972086,61
3,"{'alpha': 1e-06, 'copy_X': True, 'max_iter': 1...",-55.972086,61
4,"{'alpha': 1e-06, 'copy_X': True, 'max_iter': N...",-55.972086,61
...,...,...,...
95,"{'alpha': 20, 'copy_X': False, 'max_iter': 700}",-56.448279,91
96,"{'alpha': 20, 'copy_X': False, 'max_iter': 800}",-56.448279,91
97,"{'alpha': 20, 'copy_X': False, 'max_iter': 900}",-56.448279,91
98,"{'alpha': 20, 'copy_X': False, 'max_iter': 1000}",-56.448279,91


In [ ]:
ridge_reg_gridcv.best_estimator_

Ridge(alpha=1, max_iter=700, random_state=42)

In [ ]:
# define ridge regression model
from sklearn.linear_model import Ridge

ridge_reg = Ridge(random_state=42, alpha=1, max_iter=700)

ridge_reg.fit(X_train, y_train)

Ridge(alpha=1, max_iter=700, random_state=42)

In [ ]:
# # define ridge regression model
# from sklearn.linear_model import Ridge

# ridge_reg = Ridge(random_state=42)

# # tune lambda (denote as 'alpha' in sklearn)
# # using GridSearchCV
# from sklearn.model_selection import GridSearchCV

# # hyperparameter values we want to tune
# parameters = {
#     'alpha': (0.000001,0.00001,0.0001,0.001,
#               0.01, 0.1, 1, 5, 10, 20),
#     'max_iter' : (700,800,900,1000, None),
#     'copy_X': (True, False)
# }

# # the tuning
# ridge_reg_gridcv = GridSearchCV(ridge_reg, parameters, cv=5,
#                                 scoring='neg_root_mean_squared_error')
# ridge_reg_gridcv.fit(X_train, y_train)

In [ ]:
# tune decision tree
from sklearn.tree import DecisionTreeRegressor
tree_reg = DecisionTreeRegressor(random_state=42)
# hyperparameter values we want to tune
# ori max_depth : None -> default
# imp max_depth : 5,10,20,30,40,50
# ori vs imp

parameters = {
    'max_depth': (5,10,20,30,40,50, None),
    'criterion' : ('squared_error','friedman_mse','absolute_error','poisson'),
    'splitter' : ('best','random'),
    'max_features' : ('sqrt', 'log2', None)
}
# the tuning
tree_reg_gridcv = GridSearchCV(tree_reg, parameters, cv=5,
                                scoring='neg_root_mean_squared_error')
tree_reg_gridcv.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42),
             param_grid={'criterion': ('squared_error', 'friedman_mse',
                                       'absolute_error', 'poisson'),
                         'max_depth': (5, 10, 20, 30, 40, 50, None),
                         'max_features': ('sqrt', 'log2', None),
                         'splitter': ('best', 'random')},
             scoring='neg_root_mean_squared_error')

In [ ]:
heart.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [ ]:
# split the data
from sklearn.model_selection import train_test_split

X = heart.drop(columns='DEATH_EVENT').to_numpy()
y = heart[['DEATH_EVENT']].to_numpy()
y = y.reshape(len(y),) # sklean formatting

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
heart['DEATH_EVENT'].value_counts()

,count
DEATH_EVENT,
0,203
1,96


In [ ]:
# define the estimator/model
from sklearn.neighbors import KNeighborsClassifier

knn_clf = KNeighborsClassifier()

# hyperparameter tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# default 5 : atas dan bawah 5
parameters = {
    'n_neighbors': (2,3,4,5,6,7,8),
    'metric' : ('manhattan','euclidean','cosine','cityblock'),
    'weights' : ('uniform', 'distance')
}


knn_clf_gridcv = GridSearchCV(knn_clf, parameters, cv=5, scoring='recall')
knn_clf_gridcv.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'metric': ('manhattan', 'euclidean', 'cosine',
                                    'cityblock'),
                         'n_neighbors': (2, 3, 4, 5, 6, 7, 8),
                         'weights': ('uniform', 'distance')},
             scoring='recall')

In [ ]:
# define the estimator/model
from sklearn.tree import DecisionTreeClassifier

tree_clf = DecisionTreeClassifier()

# hyperparameter tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# default 5 : atas dan bawah 5
parameters = {
    'criterion' : ('gini','entropy','log_loss')
}


tree_clf_gridcv = GridSearchCV(tree_clf, parameters, cv=5, scoring='f1')
tree_clf_gridcv.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ('gini', 'entropy', 'log_loss')},
             scoring='f1')

In [ ]:
pd.DataFrame(tree_clf_gridcv.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001806,0.000355,0.002593,0.000665,gini,{'criterion': 'gini'},0.620690,0.521739,0.866667,0.500000,0.64,0.629819,0.130244,2
1,0.001585,0.000053,0.002133,0.000037,entropy,{'criterion': 'entropy'},0.666667,0.400000,0.785714,0.600000,0.56,0.602476,0.126889,3
2,0.001525,0.000033,0.002162,0.000166,log_loss,{'criterion': 'log_loss'},0.689655,0.461538,0.714286,0.666667,0.64,0.634429,0.089876,1


In [ ]:
# define the estimator/model
from sklearn.tree import DecisionTreeClassifier
tree_clf = DecisionTreeClassifier()
# hyperparameter tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
# default 5 : atas dan bawah 5
parameters = {
    'criterion' : ('gini','entropy','log_loss')
}
tree_clf_gridcv = RandomizedSearchCV(tree_clf, parameters, cv=5, scoring='f1')
tree_clf_gridcv.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


RandomizedSearchCV(cv=5, estimator=DecisionTreeClassifier(),
                   param_distributions={'criterion': ('gini', 'entropy',
                                                      'log_loss')},
                   scoring='f1')

In [ ]:
# the usual splitting
from sklearn.model_selection import train_test_split

X = heart.drop(columns='DEATH_EVENT').to_numpy()
y = heart[['DEATH_EVENT']].to_numpy()
y = y.reshape(len(y),)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# define random forest classifier model
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(random_state=42)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# default 5 : atas dan bawah 5
parameters = {
    'criterion' : ('gini','entropy','log_loss')
}
tree_clf_gridcv = RandomizedSearchCV(tree_clf, parameters, cv=5, scoring='f1')
tree_clf_gridcv.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


RandomizedSearchCV(cv=5, estimator=DecisionTreeClassifier(),
                   param_distributions={'criterion': ('gini', 'entropy',
                                                      'log_loss')},
                   scoring='f1')

In [ ]:
# !pip install optuna

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# n_estimators : 100
# GS dab RS : 80,90,100,110,120
# ternyata yang terbaik adalah 105

# optuna
# min dan max : 80 - 120
# akan mencari 80 - 120, -> 81, 83, 87, 111,
# n_estimators default 100

# n_estimators = trial.suggest_int('n_estimators', 50, 150)
# 'n_estimators' : (51,52,53,53,....., 148,149,150) : 150 value

# max_depth = trial.suggest_int('max_depth', 2, 32, log=True)
# 'max_depth' : (2,3,4,....., 30,31,32) : 30 value



In [ ]:
def objective(trial):
  n_estimators = trial.suggest_int('n_estimators', 50, 150)
  max_depth = trial.suggest_int('max_depth', 2, 32, log=True)
  max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

  model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, max_features=max_features)

  scores = cross_val_score(model, X, y, cv=5, scoring='f1')

  return scores.mean()

# Buat Optuna Study
study = optuna.create_study(direction='maximize')

# Jalankan proses optimasi
study.optimize(objective, n_trials=100)

# Dapatkan hasil terbaik
best_params = study.best_params
best_value = study.best_value
print(f"Best parameters: {best_params}")
print(f"Best value: {best_value}")

[I 2025-07-26 04:21:57,960] A new study created in memory with name: no-name-c4803b7e-aea4-4ede-9b19-47614ee30569
[I 2025-07-26 04:21:59,173] Trial 0 finished with value: 0.42622818622818626 and parameters: {'n_estimators': 119, 'max_depth': 8, 'max_features': None}. Best is trial 0 with value: 0.42622818622818626.
[I 2025-07-26 04:21:59,708] Trial 1 finished with value: 0.45584116366725064 and parameters: {'n_estimators': 65, 'max_depth': 2, 'max_features': 'log2'}. Best is trial 1 with value: 0.45584116366725064.
[I 2025-07-26 04:22:00,148] Trial 2 finished with value: 0.47246246246246243 and parameters: {'n_estimators': 52, 'max_depth': 14, 'max_features': 'log2'}. Best is trial 2 with value: 0.47246246246246243.
[I 2025-07-26 04:22:01,305] Trial 3 finished with value: 0.40961409151396316 and parameters: {'n_estimators': 113, 'max_depth': 19, 'max_features': None}. Best is trial 2 with value: 0.47246246246246243.
[I 2025-07-26 04:22:01,924] Trial 4 finished with value: 0.36321356128

Best parameters: {'n_estimators': 109, 'max_depth': 10, 'max_features': 'log2'}
Best value: 0.5242857142857142
